# 03 — Regression Analysis: OD vs SD

Fit a standard multiple linear regression model on each pair of **Original Data (OD)** and **Synthetic Data (SD)** datasets.  
Compare estimated coefficients side-by-side and persist the fitted models for evaluation in Step 04.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import json
import os
import glob
import pickle

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [2]:
# ── Load Configuration ──────────────────────────────────────────────────────
config_path = os.path.join("..", "config", "config.json")
with open(config_path, "r") as f:
    config = json.load(f)

print("Configuration loaded:")
print(json.dumps(config, indent=2))

Configuration loaded:
{
  "simulation": {
    "N": 1000,
    "p": 10,
    "sigma_2": 1,
    "random_seed_base": 42,
    "test_seed": 123,
    "N_test": 1000
  },
  "parameters": {
    "rho": [
      0.0,
      0.3,
      0.5
    ],
    "beta": [
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1
    ]
  },
  "synthesis": {
    "method": "cart"
  }
}


In [3]:
# ── Discover matched OD / SD file pairs ────────────────────────────────────
od_dir = os.path.join("..", "data", "original")
sd_dir = os.path.join("..", "data", "synthetic")
model_dir = os.path.join("..", "models")
os.makedirs(model_dir, exist_ok=True)

od_files = sorted(glob.glob(os.path.join(od_dir, "OD_*.csv")))
sd_files = sorted(glob.glob(os.path.join(sd_dir, "SD_*.csv")))

assert len(od_files) > 0, f"No OD files found in {od_dir}"
assert len(od_files) == len(sd_files), (
    f"Mismatch: {len(od_files)} OD vs {len(sd_files)} SD files"
)

print(f"Found {len(od_files)} OD/SD pair(s):")
for f in od_files:
    print(f"  {os.path.basename(f)}")

Found 3 OD/SD pair(s):
  OD_N1000_p10_rho0.0.csv
  OD_N1000_p10_rho0.3.csv
  OD_N1000_p10_rho0.5.csv


## Fit OLS Models

For each scenario (one per `rho` value), fit $y = X\beta + \epsilon$ on both the OD and SD datasets using ordinary least squares.

In [4]:
# ── Fit models for each OD / SD pair ───────────────────────────────────────
results = []

for od_path, sd_path in zip(od_files, sd_files):
    # Read data
    od = pd.read_csv(od_path)
    sd = pd.read_csv(sd_path)

    scenario = os.path.basename(od_path).replace("OD_", "").replace(".csv", "")
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario}")
    print(f"{'='*60}")

    # Separate features & target
    X_od, y_od = od.drop(columns="y"), od["y"]
    X_sd, y_sd = sd.drop(columns="y"), sd["y"]

    # Fit OLS on Original Data
    lm_od = LinearRegression().fit(X_od, y_od)
    r2_od = r2_score(y_od, lm_od.predict(X_od))

    # Fit OLS on Synthetic Data
    lm_sd = LinearRegression().fit(X_sd, y_sd)
    r2_sd = r2_score(y_sd, lm_sd.predict(X_sd))

    print(f"  OD model R²: {r2_od:.6f}")
    print(f"  SD model R²: {r2_sd:.6f}")

    # Build coefficient comparison table
    terms = ["(Intercept)"] + [f"X{i+1}" for i in range(X_od.shape[1])]
    beta_od = np.concatenate([[lm_od.intercept_], lm_od.coef_])
    beta_sd = np.concatenate([[lm_sd.intercept_], lm_sd.coef_])

    coef_df = pd.DataFrame({
        "term": terms,
        "beta_OD": np.round(beta_od, 6),
        "beta_SD": np.round(beta_sd, 6),
        "diff": np.round(beta_od - beta_sd, 6),
    })
    display(coef_df)

    # Save paired models
    model_path = os.path.join(model_dir, f"models_{scenario}.pkl")
    with open(model_path, "wb") as f:
        pickle.dump({"lm_od": lm_od, "lm_sd": lm_sd, "scenario": scenario}, f)
    print(f"  Saved: {os.path.basename(model_path)}")

    results.append({
        "scenario": scenario,
        "r2_od": r2_od,
        "r2_sd": r2_sd,
    })


Scenario: N1000_p10_rho0.0
  OD model R²: 0.909255
  SD model R²: 0.382186


,term,beta_OD,beta_SD,diff
0,(Intercept),1.004781,1.140148,-0.135367
1,X1,0.930015,0.495183,0.434832
2,X2,1.011114,0.765975,0.245140
3,X3,1.037114,0.747215,0.289898
4,X4,1.019297,0.989182,0.030115
5,X5,0.974563,0.675673,0.298891
6,X6,0.983320,0.467623,0.515697
7,X7,0.965138,0.624035,0.341102
8,X8,0.972380,0.538576,0.433804
9,X9,1.016764,0.571327,0.445437


  Saved: models_N1000_p10_rho0.0.pkl

Scenario: N1000_p10_rho0.3
  OD model R²: 0.970981
  SD model R²: 0.699157


,term,beta_OD,beta_SD,diff
0,(Intercept),0.971710,0.861466,0.110244
1,X1,0.987099,0.801862,0.185237
2,X2,1.058110,0.989699,0.068411
3,X3,0.977793,0.553763,0.424030
4,X4,0.985991,1.366760,-0.380768
5,X5,0.967635,0.850017,0.117618
6,X6,1.027848,0.998688,0.029161
7,X7,0.989041,0.979999,0.009042
8,X8,1.005621,0.426153,0.579469
9,X9,0.991311,0.711516,0.279795


  Saved: models_N1000_p10_rho0.3.pkl

Scenario: N1000_p10_rho0.5
  OD model R²: 0.981218
  SD model R²: 0.812854


,term,beta_OD,beta_SD,diff
0,(Intercept),0.970570,1.020032,-0.049462
1,X1,0.944131,0.902949,0.041182
2,X2,0.999182,1.217965,-0.218783
3,X3,0.972134,0.618951,0.353183
4,X4,1.101061,1.228530,-0.127470
5,X5,1.059943,0.706744,0.353199
6,X6,0.883026,0.900149,-0.017123
7,X7,0.995277,0.836060,0.159217
8,X8,0.986539,0.562757,0.423782
9,X9,0.959764,0.805785,0.153978


  Saved: models_N1000_p10_rho0.5.pkl


## Summary

In [5]:
summary_df = pd.DataFrame(results)
display(summary_df)
print("\n[DONE] Regression analysis complete. Models saved to ../models/")

,scenario,r2_od,r2_sd
0,N1000_p10_rho0.0,0.909255,0.382186
1,N1000_p10_rho0.3,0.970981,0.699157
2,N1000_p10_rho0.5,0.981218,0.812854



[DONE] Regression analysis complete. Models saved to ../models/
